In [ ]:
##### 1
from google_auth_oauthlib.flow import InstalledAppFlow

# ============================================
# Google OAuth 2.0 Authorization Script
# Scopes: Google Calendar + Gmail Send
# ============================================

# Step 1: Define the scopes required for the application
# 'gmail.send' allows sending emails via the Gmail API
SCOPES = [
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/gmail.send"
]

# Step 2: Initialize the OAuth 2.0 flow using the credentials file
flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)

# Step 3: Launch the local server to complete the OAuth flow
# A browser window will open for user login and consent
creds = flow.run_local_server(port=8080)

#  Print confirmation and access token (for debugging only)
print("\n Authorization Successful — Gmail Send Scope Included")
print("Access Token:", creds.token)

In [11]:
##### 2
import os
import pickle
from google.auth.transport.requests import Request

# ============================================
# Google OAuth 2.0 Token Handling
# ============================================

# Check if existing credentials are valid or need refresh
if creds and creds.valid:
    print(" Token is valid — no need to reauthorize.")
elif creds and creds.expired and creds.refresh_token:
    creds.refresh(Request())
else:
    # If no valid credentials, start a new authorization flow
    flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
    creds = flow.run_local_server(port=8080)

#  Save credentials securely for future use
with open("token.pkl", "wb") as token:
    pickle.dump(creds, token)

print(" Token saved successfully!")

 Token is valid — no need to reauthorize.
 Token saved successfully!


In [12]:
#### 3
import pickle
import os.path
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

# ------------------------------------------------------------
# Global Service Objects (GMAIL_SERVICE is required for sending emails)
# ------------------------------------------------------------
CALENDAR_SERVICE = None
GMAIL_SERVICE = None

# ------------------------------------------------------------
# 1. Load Saved Token
# ------------------------------------------------------------
creds = None
if os.path.exists('token.pkl'):
    with open('token.pkl', 'rb') as token:
        creds = pickle.load(token)

# ------------------------------------------------------------
# 2. Refresh Token (if necessary)
# ------------------------------------------------------------
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        # NOTE: If 'token.pkl' is missing or invalid,
        # you need to run the token generation code first.
        print(" Token file missing or invalid. Please run the token generation code first.")
        # For now, we are only checking the 'creds' object.
        pass

# ------------------------------------------------------------
# 3. Create Calendar and Gmail Service Objects
# ------------------------------------------------------------
if creds and creds.valid:
    # Create Calendar Service Object
    CALENDAR_SERVICE = build('calendar', 'v3', credentials=creds)
    print(" Google Calendar Service Object created successfully.")
    
    # ⭐ Create Gmail Service Object (required for sending emails)
    GMAIL_SERVICE = build('gmail', 'v1', credentials=creds)
    print("✅ Google Gmail Service Object (for sending) created successfully.")
    
else:
    print(" Failed to create Google Service Objects. Please check credentials.")


 Google Calendar Service Object created successfully.
✅ Google Gmail Service Object (for sending) created successfully.


In [13]:
#### 4
from datetime import datetime, timedelta
import pytz

# ------------------------------------------------------------
# Set Timezone (Important!)
# ------------------------------------------------------------
TIMEZONE = 'Asia/Karachi'  # You can set your own local timezone
tz = pytz.timezone(TIMEZONE)

# ------------------------------------------------------------
# Fetch events from today up to the next 7 days
# ------------------------------------------------------------
now = datetime.now(tz)
timeMin = now.isoformat()
timeMax = (now + timedelta(days=7)).isoformat()

# ------------------------------------------------------------
# Fetch Calendar Events
# ------------------------------------------------------------
print(" Fetching Calendar Events...")

events_result = CALENDAR_SERVICE.events().list(
    calendarId='primary',  # Using the 'primary' calendar ID
    timeMin=timeMin,
    timeMax=timeMax,
    singleEvents=True,
    orderBy='startTime'
).execute()

events = events_result.get('items', [])

# ------------------------------------------------------------
# Display Event Details
# ------------------------------------------------------------
if not events:
    print(" No events found during this period.")
else:
    print(f" {len(events)} events found.")
    print("-" * 50)
    
    # Display details for each event
    for event in events:
        summary = event.get('summary', 'No Title')
        
        # Extract Start and End Time
        start = event['start'].get('dateTime', event['start'].get('date'))
        end = event['end'].get('dateTime', event['end'].get('date'))
        
        # Extract Description (if available)
        description = event.get('description', 'No Description available.')
        
        print(f"** Event Title:** {summary}")
        print(f"   ** Starts At:** {start}")
        print(f"   ** Ends At:** {end}")
        print(f"   ** Description:** {description[:80]}{'...' if len(description) > 80 else ''}")

        # Display Attendees (Guests)
        attendees = event.get('attendees')
        if attendees:
            print("   ** Guests (Attendees):**")
            for attendee in attendees:
                email = attendee.get('email', 'Email Not Found')
                status = attendee.get('responseStatus', 'Unknown Status')
                print(f"      - {email} (Status: {status})")
        else:
            print("   ** Guests (Attendees):** No guests invited.")
            
        print("-" * 50)


 Fetching Calendar Events...
 1 events found.
--------------------------------------------------
** Event Title:** client interview for AI internship
   ** Starts At:** 2025-10-14T16:15:00+05:00
   ** Ends At:** 2025-10-14T16:30:00+05:00
   ** Description:** Client interview for AI Internship position.  
Discussion will include backgroun...
   ** Guests (Attendees):**
      - useforworkonly111@gmail.com (Status: needsAction)
      - hamzahmuhammad750@gmail.com (Status: accepted)
--------------------------------------------------


In [14]:
#### 5
from typing import TypedDict, List
from langchain_core.messages import BaseMessage

# ------------------------------------------------------------
# Define essential information for the system state
# This data will be shared among agents.
# ------------------------------------------------------------
class GraphState(TypedDict):
    """
    Represents the state of our multi-agent system.
    Agents will read and update this state.
    """

    # 1 Raw data from Google Calendar
    calendar_events: List[dict]
    
    # 2 Processed data after analysis
    # Includes reminders and guest lists for each event
    reminders_to_send: List[dict]
    
    # 3 Record of all sent emails
    sent_emails_log: List[str]
    
    # 4 Internal message for logging or inter-agent communication
    # Commonly used by the Orchestrator Agent
    internal_message: str
    
    # 5 Next action to be taken (required for LangGraph)
    next_action: str
    
    # 6 Current error message (if any issue occurs)
    error_message: str

print(" Block 1: LangGraph State (GraphState) successfully defined.")

 Block 1: LangGraph State (GraphState) successfully defined.


In [15]:
#### 6
from langchain.tools import tool
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from datetime import datetime, timedelta
import pytz
import base64
from email.mime.text import MIMEText

# ------------------------------------------------------------
# NOTE: These global variables come from previous blocks
# (Service Initialization)
# ------------------------------------------------------------
global CALENDAR_SERVICE
global GMAIL_SERVICE
global USER_EMAIL
USER_EMAIL = 'me'  # Authorized user ID for Gmail API

# ------------------------------------------------------------
# Helper Function (Required for Email Encoding)
# ------------------------------------------------------------
def create_mime_message(sender: str, to: str, subject: str, body: str) -> Dict[str, str]:
    """
    Prepares an email message in Base64-encoded MIME format.
    """
    # Body is kept in HTML format
    message = MIMEText(body, 'html', 'utf-8')
    message['to'] = to
    message['from'] = sender
    message['subject'] = subject

    encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
    return {'raw': encoded_message}

# ------------------------------------------------------------
# Pydantic Schemas for Tools
# ------------------------------------------------------------
class CalendarEventInput(BaseModel):
    """Input schema for fetching calendar events (kept simple)."""
    pass

class SendEmailInput(BaseModel):
    """Input schema for sending an email reminder."""
    recipient_email: str = Field(description="The email address to send the reminder to.")
    subject: str = Field(description="Email subject line.")
    body: str = Field(description="Full email text body.")

# ------------------------------------------------------------
# 1. Calendar Events Fetching Tool (Updated to include Organizer)
# ------------------------------------------------------------
@tool("fetch_calendar_events", args_schema=CalendarEventInput)
def fetch_calendar_events() -> List[Dict[str, Any]]:
    """
    Retrieves event data (summary, attendees, time, and organizer) from Google Calendar.
    """
    if CALENDAR_SERVICE is None:
        return [{"error": "Calendar Service not initialized.", "details": "CALENDAR_SERVICE is None"}]
    
    try:
        # Time settings (Next 7 days)
        tz = pytz.timezone('Asia/Karachi')
        now = datetime.now(tz)
        timeMin = now.isoformat()
        timeMax = (now + timedelta(days=7)).isoformat()

        events_result = CALENDAR_SERVICE.events().list(
            calendarId='primary', timeMin=timeMin, timeMax=timeMax,
            maxResults=10, singleEvents=True, orderBy='startTime'
        ).execute()
        
        events = events_result.get('items', [])
        if not events:
            return [{"status": "No Events Found", "message": "No events found in the next 7 days."}]
        
        cleaned_events = []
        for event in events:
            start = event['start'].get('dateTime', event['start'].get('date'))
            end = event['end'].get('dateTime', event['end'].get('date'))
            
            # ⭐ Organizer Email (required by LLM)
            organizer_email = event.get('organizer', {}).get('email', 'unknown@organizer.com')
            
            # Extract Attendees
            attendees = []
            for att in event.get('attendees', []):
                if att.get('responseStatus') != 'declined' and att.get('email'):
                    attendees.append({
                        'email': att.get('email'),
                        'status': att.get('responseStatus')
                    })
            
            cleaned_events.append({
                'summary': event.get('summary', 'No Title'),
                'start_time': start,
                'end_time': end,
                'description': event.get('description', 'N/A'),
                'organizer_email': organizer_email,  # ⭐ Helps LLM identify the organizer
                'attendees': attendees,
            })
            
        return cleaned_events

    except Exception as e:
        return [{"error": "Error while fetching calendar data.", "details": str(e)}]

# ------------------------------------------------------------
# 2. Email Sending Tool
# ------------------------------------------------------------
@tool("send_email_reminder", args_schema=SendEmailInput)
def send_email_reminder(recipient_email: str, subject: str, body: str) -> str:
    """
    Sends a Base64-encoded email using the Gmail API.
    """
    if GMAIL_SERVICE is None:
        return " Error: Gmail Service not initialized. Email could not be sent."
    
    try:
        # Step 1: Create the MIME message
        message = create_mime_message(
            sender=USER_EMAIL,
            to=recipient_email,
            subject=subject,
            body=body
        )
        
        # Step 2: Send the email via Gmail API
        GMAIL_SERVICE.users().messages().send(
            userId=USER_EMAIL,
            body=message
        ).execute()
        
        return f" TOOL: Email sent successfully (REAL API): To: {recipient_email}, Subject: {subject}"

    except Exception as e:
        return f" TOOL: Failed to send email. Error: {str(e)}"

# ------------------------------------------------------------
# List of Tools
# ------------------------------------------------------------
AGENT_TOOLS = [fetch_calendar_events, send_email_reminder]

print(" Block 2: Agent Tools updated with Organizer Email field.")


 Block 2: Agent Tools updated with Organizer Email field.


In [16]:
#### 7
# ============================================
# 🔹 Block 3: LLM Initialization
# Description:
#   This block initializes the Large Language Model (LLM)
#   using the ChatOllama interface with the model "gpt-oss:120b-cloud".
#   The temperature is set to 0 for deterministic (stable) responses.
# ============================================

from langchain_community.chat_models import ChatOllama

# Initialize the LLM (Language Model)
llm = ChatOllama(
    model="gpt-oss:120b-cloud",  # Model name
    temperature=0                # 0 = less randomness (more focused output)
)

# Confirmation message
print(" Block 3: LLM (gpt-oss:120b-cloud) has been successfully initialized.")


 Block 3: LLM (gpt-oss:120b-cloud) has been successfully initialized.


C:\Users\hamza\AppData\Local\Temp\ipykernel_6092\285182653.py:13: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(


In [17]:
#### 8
# ===========================================================
# Block 4: GraphState Definition
# Description:
#   Defines the shared state structure used by multiple agents
#   to communicate and coordinate actions in the LangGraph system.
# ===========================================================

from typing import TypedDict, List, Dict, Any
from langchain_core.messages import BaseMessage

#  GraphState - Shared state between agents
class GraphState(TypedDict):
    # List of raw calendar events fetched from the API
    calendar_events: List[dict]

    # Summary generated by Agent 1 (Data Fetcher)
    analysis_summary: str

    # Draft reminder created by Agent 2 (Analyzer)
    reminder_draft: Dict[str, Any]

    # Final reminder ready to be sent by Agent 3 (Mailer)
    final_reminder_to_send: Dict[str, Any]

    # Log of successfully sent emails
    sent_emails_log: List[str]

    # Internal communication message used between agents
    internal_message: str

    # Indicates the next action to perform in the workflow
    next_action: str

    # Error message if any issue occurs during execution
    error_message: str

print(" Block 4: Tools defined successfully.")


 Block 4: Tools defined successfully.


In [18]:
#### 9
# ===========================================================
# Block 5: Agent Prompt Definitions
# Description:
#   This section defines prompt templates for three specialized agents:
#   1 Data Fetcher Agent
#   2 Reminder Draft Agent
#   3 Mailer Agent
#   Each agent uses the same LLM (defined earlier) with distinct roles.
# ===========================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatOllama
from typing import List, Dict, Any
import json
from langchain_core.runnables import Runnable

# NOTE: The `llm` (ChatOllama) model is already defined in Block 3.

# -----------------------------------------------------------
#  1. Data Fetcher Agent — Fetches and summarizes event data
# -----------------------------------------------------------
DATA_FETCHER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", 
     """You are a skilled Google Calendar Data Fetcher and Summarizer.
     **Your Role:** Analyze raw calendar event data and produce a detailed summary.
     **Your Goal:** Review fetched calendar events and print a clear, plain-text summary 
     under 'analysis_summary' (e.g., total guests, status, and timings).
     **Output:** Only plain text summary (no JSON)."""),
    
    ("human", 
     "Analyze the following events and provide a useful summary:\n{input_data}")
])

# -----------------------------------------------------------
# 🧩 2. Reminder Draft Agent — Creates tailored email reminders
# -----------------------------------------------------------
REMINDER_DRAFT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", 
     """You are an expert Reminder Draft Specialist.
     **Your Role:** Review the provided event summary and raw data to create 
     customized reminder drafts for each guest and organizer.
     
     **Important Instructions:**
     1. Always check the field **'organizer_email'** in the raw data.
     2. If the recipient’s email matches **'organizer_email'**, 
        create a reminder for the Organizer 
        (e.g., "You have your [Event Title] meeting today. Please be prepared.").
     3. If the recipient’s email differs from the organizer (and is in the attendee list),
        create a reminder for the Participant.
     
     **Goal:** For each person (organizer and participants), create a reminder draft 
     that includes `email`, `subject`, and `body`.
     **Output:** A valid **JSON array** where each object represents a complete reminder draft."""),
    
    ("human", 
     "Event Summary: {summary}\nRaw Data: {raw_data}\nPlease prepare the reminder drafts.")
])

# -----------------------------------------------------------
#  3. Mailer Agent — Processes and sends all reminder drafts
# -----------------------------------------------------------
MAILER_AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", 
     """You are a responsible Email Sender.
     **Your Role:** Review the list of reminder drafts and prepare them for sending.
     **Your Goal:** For every draft in the list, extract 'email', 'subject', 
     and 'body' fields and directly call the **'send_email_reminder'** tool.
     Skip any extra validation — send all reminders regardless of conditions."""),
    
    ("human", 
     "Here is the list of reminder drafts to process and send:\n{drafts_list}")
])

# -----------------------------------------------------------
#  Convert prompts into runnable LLM agent objects
# -----------------------------------------------------------
DATA_FETCHER_AGENT = DATA_FETCHER_PROMPT | llm
REMINDER_DRAFT_AGENT = REMINDER_DRAFT_PROMPT | llm
MAILER_AGENT = MAILER_AGENT_PROMPT | llm  # Tool calls will be handled separately

print(" Block 5: Agents updated with refined roles and improved clarity for email reminder processing.")


 Block 5: Agents updated with refined roles and improved clarity for email reminder processing.


In [19]:
#### 10
# ===========================================================
# Block 6: LangGraph Nodes and Execution Flow
# Description:
#   This section defines the main agent nodes and orchestration
#   logic for the multi-agent system following the A2A Protocol.
#
#   Agents Included:
#     1 Data Fetcher Agent  → Fetch and summarize Google Calendar events
#     2 Reminder Draft Agent → Generate personalized email drafts
#     3 Mailer Agent         → Send emails to all recipients
# ===========================================================

from langgraph.graph import StateGraph, END
from typing import Dict, Any, List
import json  # Required for parsing and handling JSON data

# NOTE:
# - Tools (`fetch_calendar_events`, `send_email_reminder`)
# - State (`GraphState`)
# - Agents (`DATA_FETCHER_AGENT`, `REMINDER_DRAFT_AGENT`)
# are all defined in earlier blocks.

# -----------------------------------------------------------
#  A. Data Fetcher Node (Agent 1)
# -----------------------------------------------------------
def run_data_fetcher(state: GraphState) -> Dict[str, Any]:
    """Fetches calendar data and generates a summary."""
    print(" Data Fetcher Agent (Summary) running...")

    events_data = fetch_calendar_events({})  # Tool call
    input_data = json.dumps(events_data, indent=2)
    summary_response = DATA_FETCHER_AGENT.invoke({"input_data": input_data})

    new_state = {
        "calendar_events": events_data,
        "analysis_summary": summary_response.content,
        "internal_message": "Event data fetched successfully. Next step: Reminder Draft.",
        "next_action": "DRAFT_REMINDER"
    }
    return new_state


# -----------------------------------------------------------
#  B. Reminder Draft Node (Agent 2)
# -----------------------------------------------------------
def run_reminder_draft(state: GraphState) -> Dict[str, Any]:
    """Prepares reminder drafts based on event summaries."""
    print(" Reminder Draft Agent (Drafting) running...")

    draft_response = REMINDER_DRAFT_AGENT.invoke({
        "summary": state['analysis_summary'],
        "raw_data": json.dumps(state['calendar_events'])
    })

    try:
        # Expected output: JSON array containing drafts
        draft_list = json.loads(draft_response.content.strip())

        new_state = {
            "reminder_draft": draft_list,
            "internal_message": f"{len(draft_list)} drafts created. Next step: Mailer Agent validation.",
            "next_action": "MAILER_VALIDATE"
        }
    except Exception as e:
        return {"error_message": f"Drafting Failed: {str(e)}", "next_action": "ERROR"}

    return new_state


# -----------------------------------------------------------
#  C. Mailer Agent Node (Agent 3)
# -----------------------------------------------------------
def run_mailer_agent(state: GraphState) -> Dict[str, Any]:
    """Processes drafts and sends emails to each recipient."""
    print(" Mailer Agent (Send All) running...")

    drafts_list = state.get('reminder_draft', [])
    reminders_sent_log = []

    # Loop through each draft and send an email
    for draft in drafts_list:
        sent_subject = draft.get('subject', 'Default Reminder Subject')
        sent_body = draft.get('body', 'Default Reminder Body')
        recipient = draft.get('email', 'test@example.com')

        print(f"\n---  REMINDER SENT (Target: {recipient}) ---")
        print(f"Subject: {sent_subject}")
        print(f"Body: {sent_body}")
        print("-----------------------------------------------------")

        try:
            # Tool call using .invoke() with dictionary arguments
            log_message = send_email_reminder.invoke({
                "recipient_email": recipient,
                "subject": sent_subject,
                "body": sent_body
            })

            # Append successful send details to log
            reminders_sent_log.append({
                "recipient": recipient,
                "subject": sent_subject,
                "log": log_message
            })

        except Exception as e:
            reminders_sent_log.append({
                "recipient": recipient,
                "subject": sent_subject,
                "log": f"Email failed (Tool Invocation Error): {str(e)}"
            })

    new_state = {
        "sent_emails_log": state.get("sent_emails_log", []) + reminders_sent_log,
        "internal_message": f"Mailer Agent processed {len(reminders_sent_log)} emails. Task completed.",
        "next_action": "FINISH"
    }
    return new_state


# -----------------------------------------------------------
#  D. Orchestrator Node (Flow Controller)
# -----------------------------------------------------------
def route_orchestrator(state: GraphState) -> str:
    """Determines the next agent to execute, similar to A2A protocol."""
    action = state.get("next_action", "FETCH_EVENTS")
    print(f" Orchestrator: Next step -> {action}")
    return action


print(" Block 6: LangGraph nodes finalized with consistent structure and tool-call handling.")


 Block 6: LangGraph nodes finalized with consistent structure and tool-call handling.


In [20]:
#### 11
#  Block 7: LangGraph Workflow Compilation

from langgraph.graph import StateGraph, END, START

# 1 Initialize Graph State
workflow = StateGraph(GraphState)

# 2 Add Nodes (Agents)
workflow.add_node("ORCHESTRATOR", route_orchestrator)
workflow.add_node("FETCH_EVENTS", run_data_fetcher)
workflow.add_node("DRAFT_REMINDER", run_reminder_draft)
workflow.add_node("MAILER_VALIDATE", run_mailer_agent)
# Optional: You can add an error handler node if needed
# workflow.add_node("ERROR_HANDLER", run_error_handler)

# 3 Define Entry Point
# The process starts with the Data Fetcher (like A2A Protocol flow)
workflow.set_entry_point("FETCH_EVENTS")

# 4 Define Edges (Agent-to-Agent Flow)
# After fetching data → go to draft reminder agent
workflow.add_edge("FETCH_EVENTS", "DRAFT_REMINDER")

# After drafting → go to mailer agent
workflow.add_edge("DRAFT_REMINDER", "MAILER_VALIDATE")

# After mailer agent → finish workflow
workflow.add_edge("MAILER_VALIDATE", END)

# 5 Compile the Graph
app = workflow.compile()

print(" Block 7: LangGraph Workflow compiled successfully.")


 Block 7: LangGraph Workflow compiled successfully.


In [22]:
#### 12
#  Initialize Initial State
initial_state = {
    "calendar_events": [],
    "analysis_summary": "",
    "reminder_draft": {},
    "sent_emails_log": [],
    "internal_message": "System starting... Events will be fetched.",
    "next_action": "FETCH_EVENTS",
    "error_message": "",
}

print(" Starting LangGraph Agent System...")

# Invoke the Graph (Run the Workflow)
# This call starts the system from the 'FETCH_EVENTS' node
final_state = app.invoke(initial_state)

#  Display Final Output
print("\n--- ✅ Final System Output ---")
print(f"Final Message: {final_state['internal_message']}")
print(f"Sent Emails Log: {final_state['sent_emails_log']}")
print("----------------------------------")

 Starting LangGraph Agent System...
 Data Fetcher Agent (Summary) running...
 Reminder Draft Agent (Drafting) running...
 Mailer Agent (Send All) running...

---  REMINDER SENT (Target: hamzahmuhammad750@gmail.com) ---
Subject: Reminder: Client interview for AI internship today at 16:15 (UTC+5)
Body: Hello Hamzah,

This is a reminder that you have the **Client interview for AI internship** scheduled for today, 14 Oct 2025, from **16:15 to 16:30 (UTC +5)**.

**Key points to prepare:**
- Review the candidate’s background, AI/ML project experience, and internship expectations.
- Ensure a stable internet connection and that your microphone and camera are working before the start time.
- Follow up with the pending attendee (useforworkonly111@gmail.com) to confirm their participation.

Please be ready to start promptly at 16:15.

Best regards,
Your Calendar Reminder System
-----------------------------------------------------

---  REMINDER SENT (Target: useforworkonly111@gmail.com) ---
Subj